### Baywatch – Lösung Schritt für Schritt

*Jugendwettbewerb Informatik 2018, 3. Runde, Junioraufgabe 2 (= 37. Bundeswettbewerb Informatik, 1. Runde, Junioraufgabe 2)*

Auf ihren Fahrten über die Weltmeere ist die Piratin Longstock zur Inselgruppe der Amöben gelangt. An den Küsten dieser
grob kreisförmigen Inseln wechseln sich Buchten mit Landzungen ab. Longstock sucht nun das hipste Hauptquartier: eine Bucht
in bester Landzungen-Nachbarschaft. Aber in ihren alten Landkarten ist nur für einige Landzungen deren Beschaffenheit
angegeben.

Hier ist die Landzungen-Liste für die Insel „Big Amoeba“, von der nördlichsten Landzunge aus im Uhrzeigersinn
(`?` heißt: unbekannt):

    ? Wald ? ? ? ? Wald See ? Wiese ? ? ?

Longstock muss es aber genauer wissen! Zu jeder Insel schickt sie ihren sprechenden Papagei George. Der fliegt die
Landzungen im Uhrzeigersinn ab und prägt sich deren Beschaffenheit der Reihe nach ein. Dann kehrt er zum Schiff zurück und
erstattet Bericht. Nach seinem Flug zu Big Amoeba krächzt er erschöpft diese Landzungen-Liste:

    Wald Wald Wiese Häuser Wüste Wald See Wald Wiese Sumpf Wüste See Häuser

Dummerweise kennt George die Himmelsrichtungen nicht. Die Piratin kann also nicht wissen, bei welcher Landzunge George
gestartet ist. Kann Longstock trotzdem ihre Karte vervollständigen?

#### Junioraufgabe 2

Schreibe ein Programm, das zwei Landzungen-Listen einer Amöben-Insel einliest: eine lückenhafte (mit Beginn im Norden) und
eine vollständige (aber mit unbekanntem Beginn). Das Programm soll eine passende vollständige Landzungen-Liste ausgeben,
die mit der nördlichen Landzunge beginnt und die Landzungen im Uhrzeigersinn durchläuft.

Wende dein Programm auf die Beispiele an, die du auf den BWINF-Webseiten findest.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `baywatch1.txt` bis `baywatch6.txt`. Jede Datei hat zwei Zeilen:

- 1. Zeile: die vollständige Liste von George, jede Landzunge als Ziffer: 1 Wald, 2 Wiese, 3 Häuser, 4 Wüste, 5 See, 6 Sumpf, 7 Reisfeld,
  8 Berg, 9 Vulkankrater
- 2. Zeile: die lückenhafte Liste von Longstock, mit `?` für unbekannte Landzungen

`baywatch1.txt` ist das Beispiel „Big Amoeba“.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Kreis als Liste:** Die Landzungen liegen im Kreis, wir speichern sie als Liste. Der Rest des Kreises nach dem Ende der Liste beginnt wieder vorn.
  Mit **Modulo** (`%`) rechnet man im Kreis: `(i + j) % n`.
- **Verschieben (Rotation):** Georgs Liste beginnt an einer unbekannten Stelle. Wir probieren alle `n` möglichen Anfangspunkte durch.
- **Muster mit Platzhaltern vergleichen:** Ein `?` passt zu allem, jede andere Landzunge nur zu sich selbst.
- **Lineare Suche über alle Möglichkeiten (Brute Force):** Für jede Verschiebung vergleicht man alle `n` Positionen. Das sind `n²` Vergleiche,
  bei 200 oder 500 Landzungen noch kein Problem.
- **Abbruch bei der ersten Abweichung:** Sobald eine Position nicht passt, ist die Verschiebung falsch, und man probiert die nächste.

**Nützliche Funktionen** (Bibliotheken braucht man keine)
- `liste.split()`, `len`, `range`, `%`, `in`, `liste[a:] + liste[:a]` (Liste um `a` Stellen rotieren), `' '.join(...)`, `str(zahl)` und `int(text)`

## Die Idee

Georgs Liste `G` hat `n` Landzungen, und Longstocks Karte `K` hat ebenfalls `n` Einträge (von denen manche `?` sind).
Nur die Liste von George ist vollständig, nur sie hat die richtigen Landzungen. Wir suchen eine **Verschiebung** `v` zwischen 0 und `n − 1`, mit der
die Karte zur Liste passt:

`K[j]` ist `?` **oder** `K[j] == G[(v + j) % n]` für alle Positionen `j` von 0 bis `n − 1`.

Dann ist die gesuchte vollständige Liste, die im Norden beginnt, genau `G[(v + j) % n]` für `j = 0, 1, ..., n − 1`. Das ist Georgs Liste, die so
verschoben ist, dass sie dort beginnt, wo der Norden ist.

Beispiel `baywatch1.txt`: Georgs Liste hat 13 Einträge. Ab der Verschiebung 12 beginnt sie mit der 3 (Häuser, dem letzten Eintrag), dann folgen
die anderen: 3 1 1 2 3 4 1 5 1 2 6 4 5. Das passt zur Karte `? 1 ? ? ? ? 1 5 ? 2 ? ? ?`.

## Schritt 1: Die Datei einlesen

`readline()` liest eine Zeile, `split()` zerlegt sie in einzelne Wörter, hier die Ziffern oder das Fragezeichen. Georgs Liste
wandeln wir mit `int(...)` in Zahlen um. In Longstocks Karte lassen wir das Fragezeichen als Text `'?'` stehen und wandeln die Ziffern ebenfalls
in Zahlen um.

In [ ]:
def lies_karten(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        georg_zeile = f.readline().split()
        karte_zeile = f.readline().split()
    georg = []
    for text in georg_zeile:
        georg.append(int(text))
    karte = []
    for text in karte_zeile:
        if text == '?':
            karte.append('?')
        else:
            karte.append(int(text))
    return georg, karte


georg, karte = lies_karten('beispieldaten/baywatch1.txt')
print('George:', georg)
print('Karte: ', karte)

George: [1, 1, 2, 3, 4, 1, 5, 1, 2, 6, 4, 5, 3]
Karte:  ['?', 1, '?', '?', '?', '?', 1, 5, '?', 2, '?', '?', '?']


## Schritt 2: Passt eine Verschiebung?

Die Funktion `passt(georg, karte, v)` vergleicht alle Positionen. Für jede Position `j` ist der passende Eintrag bei George `georg[(v + j) % n]`.
Das `% n` sorgt dafür, dass der Index am Ende der Liste wieder bei 0 anfängt (Kreis).

Sobald eine Position nicht passt (die Karte hat kein `?` und die Zahlen sind verschieden), geben wir sofort `False` zurück. Nur wenn alle Positionen
passen, ist es `True`.

In [ ]:
def passt(georg, karte, v):
    n = len(georg)
    for j in range(n):
        if karte[j] != '?' and karte[j] != georg[(v + j) % n]:
            return False
    return True


print(passt(georg, karte, 0), passt(georg, karte, 1), passt(georg, karte, 12))

False False True


## Schritt 3: Alle Verschiebungen probieren

`finde_verschiebungen` probiert alle Verschiebungen von 0 bis `n − 1` aus und sammelt die, die passen. Die Aufgabe verlangt nur **eine** Lösung, aber es kann mehrere
geben (wenn die Karte zum Beispiel nur Fragezeichen hat). Wir sammeln alle und geben die erste aus.

`loesung_liste(georg, v)` baut die vollständige Liste im Norden-Beginn. Statt mit der Schleife geht es auch kürzer mit einem Ausschnitt:
`georg[v:] + georg[:v]` hängt den Teil ab `v` an den Anfang und den Teil davor ans Ende.

In [ ]:
def finde_verschiebungen(georg, karte):
    passende = []
    for v in range(len(georg)):
        if passt(georg, karte, v):
            passende.append(v)
    return passende


def loesung_liste(georg, v):
    return georg[v:] + georg[:v]


verschiebungen = finde_verschiebungen(georg, karte)
print('Passende Verschiebungen:', verschiebungen)
print('Lösung:', loesung_liste(georg, verschiebungen[0]))

Passende Verschiebungen: [12]
Lösung: [3, 1, 1, 2, 3, 4, 1, 5, 1, 2, 6, 4, 5]


## Schritt 4: Mit Namen ausgeben

Die Ziffern stehen für Landschaften. Für `baywatch1.txt` geben wir die Lösung auch mit den Namen aus. Mit der Liste `NAMEN` ist `NAMEN[ziffer - 1]` der Name zur Ziffer.
Für die Dokumentation geben wir die Verschiebungen und die erste Lösung aus (bei langen Listen die ersten und letzten Einträge).

In [ ]:
NAMEN = ['Wald', 'Wiese', 'Häuser', 'Wüste', 'See', 'Sumpf', 'Reisfeld', 'Berg', 'Vulkankrater']


def in_text(liste):
    teile = []
    for eintrag in liste:
        teile.append(str(eintrag))
    return ' '.join(teile)


def bearbeite(dateiname):
    georg, karte = lies_karten(dateiname)
    fragezeichen = karte.count('?')
    verschiebungen = finde_verschiebungen(georg, karte)
    print(f'{dateiname}: {len(georg)} Landzungen, davon {fragezeichen} unbekannt')
    if len(verschiebungen) == 0:
        print('  Keine passende Landzungen-Liste gefunden.')
        return
    print(f'  Passende Verschiebungen: {len(verschiebungen)}, erste: {verschiebungen[0]}')
    loesung = loesung_liste(georg, verschiebungen[0])
    if len(loesung) <= 20:
        print('  Lösung:', in_text(loesung))
        namen = []
        for ziffer in loesung:
            namen.append(NAMEN[ziffer - 1])
        print('  Das sind:', ' '.join(namen))
    else:
        print('  Lösung, die ersten 30 Landzungen:', in_text(loesung[:30]), '...')
        print('  Lösung, die letzten 10 Landzungen:', in_text(loesung[-10:]))


bearbeite('beispieldaten/baywatch1.txt')

beispieldaten/baywatch1.txt: 13 Landzungen, davon 9 unbekannt
  Passende Verschiebungen: 1, erste: 12
  Lösung: 3 1 1 2 3 4 1 5 1 2 6 4 5
  Das sind: Häuser Wald Wald Wiese Häuser Wüste Wald See Wald Wiese Sumpf Wüste See


## Schritt 5: Alle Beispiele

In [ ]:
for nr in range(1, 7):
    bearbeite(f'beispieldaten/baywatch{nr}.txt')
    print()

beispieldaten/baywatch1.txt: 13 Landzungen, davon 9 unbekannt
  Passende Verschiebungen: 1, erste: 12
  Lösung: 3 1 1 2 3 4 1 5 1 2 6 4 5
  Das sind: Häuser Wald Wald Wiese Häuser Wüste Wald See Wald Wiese Sumpf Wüste See

beispieldaten/baywatch2.txt: 200 Landzungen, davon 193 unbekannt
  Passende Verschiebungen: 1, erste: 148
  Lösung, die ersten 30 Landzungen: 3 9 6 1 1 8 7 7 7 7 6 7 9 4 3 7 7 8 9 5 1 9 5 4 8 6 2 5 7 5 ...
  Lösung, die letzten 10 Landzungen: 9 6 9 9 1 1 6 1 9 3

beispieldaten/baywatch3.txt: 200 Landzungen, davon 200 unbekannt
  Passende Verschiebungen: 200, erste: 0
  Lösung, die ersten 30 Landzungen: 2 1 3 2 8 2 2 8 8 1 4 1 3 7 6 7 1 5 4 5 6 3 4 1 4 2 5 6 9 9 ...
  Lösung, die letzten 10 Landzungen: 1 8 3 4 6 1 6 1 9 5

beispieldaten/baywatch4.txt: 200 Landzungen, davon 198 unbekannt
  Passende Verschiebungen: 2, erste: 150
  Lösung, die ersten 30 Landzungen: 3 9 6 1 1 8 7 7 7 7 6 7 9 4 3 7 7 8 9 5 1 9 5 4 8 6 2 5 7 5 ...
  Lösung, die letzten 10 Landzungen: 9 6 9 

Das stimmt mit den offiziellen Lösungshinweisen überein:

- `baywatch1`: genau eine Lösung, Verschiebung 12.
- `baywatch2`: genau eine Lösung, Verschiebung 148.
- `baywatch3`: Die Karte besteht nur aus Fragezeichen, deshalb passt jede der 200 Verschiebungen. Die erste ist 0 (Georgs Liste unverändert).
- `baywatch4`: zwei Lösungen (Verschiebungen 150 und 156). Die Hinweise zeigen die mit 150.
- `baywatch5` und `baywatch6`: genau eine Lösung (Verschiebung 150 und 64).

Bei mehreren Lösungen kann man nicht entscheiden, welche die richtige Insel ist. Das Programm gibt eine davon aus (die laut Aufgabe genügt)
und nennt auch die Anzahl der Lösungen.

## Für Neugierige: Wo steht der Norden?

Das Beispiel `baywatch4.txt` hat zwei Lösungen. Beide stimmen an den bekannten Landzungen mit der Karte überein und unterscheiden sich nur dort, wo die Karte Fragezeichen hat.
Wir zeigen beide Verschiebungen und prüfen das nach.

In [ ]:
georg, karte = lies_karten('beispieldaten/baywatch4.txt')
verschiebungen = finde_verschiebungen(georg, karte)
print('Verschiebungen:', verschiebungen)
for v in verschiebungen:
    loesung = loesung_liste(georg, v)
    bekannte_stellen = 0
    stimmen = True
    for j in range(len(karte)):
        if karte[j] != '?':
            bekannte_stellen = bekannte_stellen + 1
            if loesung[j] != karte[j]:
                stimmen = False
    print(f'Verschiebung {v}: {bekannte_stellen} bekannte Stellen, alle stimmen: {stimmen}')

l1 = loesung_liste(georg, verschiebungen[0])
l2 = loesung_liste(georg, verschiebungen[1])
unterschiede = 0
for j in range(len(l1)):
    if l1[j] != l2[j]:
        unterschiede = unterschiede + 1
print('Die beiden Lösungen unterscheiden sich an', unterschiede, 'von', len(l1), 'Stellen.')

Verschiebungen: [150, 156]
Verschiebung 150: 2 bekannte Stellen, alle stimmen: True
Verschiebung 156: 2 bekannte Stellen, alle stimmen: True
Die beiden Lösungen unterscheiden sich an 166 von 200 Stellen.


Für sehr lange Listen (Millionen von Landzungen) wäre das Verfahren mit `n²` Vergleichen zu langsam. Dann benutzt man Verfahren aus der Textsuche, zum Beispiel
den Knuth-Morris-Pratt-Algorithmus oder die Fast-Fourier-Transformation für das Suchen mit Platzhaltern. Das ist weit über die Aufgabe hinaus.